# CUHK-X Large Model Track — Colab 실행 노트북

워크플로: **[로컬 VS Code] 코드 수정 → git push → [이 노트북] Runtime > Run all**

- 모든 셀은 **재실행 가능(idempotent)** — VM이 리셋되면 그냥 위에서부터 다시 실행
- 결과물(submission/val_pred/전처리 캐시)은 Drive에 저장 → 세션이 끊겨도 resume 됨
- 런타임 유형: **L4 이상**(7B), T4면 아래 MODEL을 3B로

### 최초 1회 준비 (HF gated dataset 인증)

1. [HF 데이터셋 페이지](https://huggingface.co/datasets/Kevin-Pal/CUHK-X_Large_Model_Track)에서 로그인 후 **Agree and access repository** 클릭 (승인제면 승인까지 대기)
2. [HF 토큰 설정](https://huggingface.co/settings/tokens)에서 **Read** 토큰 발급
3. Colab 왼쪽 사이드바 🔑 **보안 비밀(Secrets)**에 이름 `HF_TOKEN`으로 토큰 저장 + 노트북 액세스 허용

In [ ]:
# ===== 0. 설정 =====
REPO_URL  = "https://github.com/jangs03/cuhk_euron.git"
DRIVE_OUT = "/content/drive/MyDrive/cuhk"          # 결과 저장 위치 (Drive)
MODEL     = "Qwen/Qwen2.5-VL-7B-Instruct"          # T4(16GB)면 3B로 변경
MODALITY  = "IR"                                    # IR이 가장 선명 (Depth_Color도 실험 가치)

!nvidia-smi -L

In [ ]:
# ===== 1. Drive 마운트 (결과 저장/resume 용) =====
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
# ===== 2. 코드 받기 (없으면 clone, 있으면 pull) =====
import os
if not os.path.exists('/content/cuhk_euron'):
    !git clone {REPO_URL} /content/cuhk_euron
%cd /content/cuhk_euron
!git pull

In [ ]:
# ===== 3. 의존성 설치 =====
!pip install -q -r requirements.txt huggingface_hub

In [ ]:
# ===== 4. 데이터 다운로드 (HF → VM 로컬 디스크) + 압축 해제 =====
# ⚠️ gated dataset이라 HF 인증 필요. 최초 1회 준비:
#   1) https://huggingface.co/datasets/Kevin-Pal/CUHK-X_Large_Model_Track 접속
#      → 로그인 후 "Agree and access repository"로 접근 동의/신청
#   2) https://huggingface.co/settings/tokens 에서 Read 토큰 발급
#   3) Colab 왼쪽 🔑(보안 비밀)에 이름 `HF_TOKEN`으로 저장 + 이 노트북의 액세스 허용
import os

try:  # Colab 보안 비밀에서 토큰 로드
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass  # 로컬 실행이거나 이미 환경변수/hf auth login으로 인증된 경우

from huggingface_hub import snapshot_download
snapshot_download('Kevin-Pal/CUHK-X_Large_Model_Track', repo_type='dataset',
                  local_dir='data', token=os.environ.get('HF_TOKEN'))

# zip은 어디에 있든 전부 data/ 바로 아래로 해제 (unzip -n: 이미 풀린 건 skip)
!find data -name "*.zip" -exec unzip -qn {} -d data \;

# csv를 data/ 루트로 복사 → 스크립트 기본 인자와 일치
import glob, shutil
for name in ['training_qa.csv', 'test_qa.csv', 'modality_list.csv']:
    hits = [h for h in glob.glob(f'data/**/{name}', recursive=True)
            if os.path.dirname(h) != 'data']
    if hits and not os.path.exists(f'data/{name}'):
        shutil.copy(hits[0], f'data/{name}')

!ls data

In [ ]:
# ===== 5. 전처리 + 결과물 Drive 저장/복원 =====
# 캐시(프레임 JPEG 수천 장)는 Drive에 직접 쓰면 매우 느림 → tar 하나로 묶어 백업/복원
import os

CACHE_TAR = f"{DRIVE_OUT}/cache.tar"
DONE_MARKS = ['data/cache/harn_index.csv', 'data/cache/hau_index.csv',
              'data/cache/test_index.csv']

# (a) 이전 세션의 전처리 결과가 Drive에 있으면 복원 (재전처리 생략)
if not os.path.exists('data/cache') and os.path.exists(CACHE_TAR):
    print('Drive에서 전처리 캐시 복원 중...')
    !tar -xf {CACHE_TAR}

# (b) 전처리 (인덱스 csv 3개가 모두 있으면 완료된 것으로 보고 skip)
if not all(os.path.exists(p) for p in DONE_MARKS):
    !python src/preprocess_harn.py --harn-root data/HARn --out data/cache/HARn \
        --index data/cache/harn_index.csv --frames 16 --workers 4 --modality {MODALITY}
    !python src/preprocess_harn.py --harn-root data/HAU --out data/cache/HAU \
        --index data/cache/hau_index.csv --frames 16 --workers 4 --modality {MODALITY}
    !python src/preprocess_harn.py --harn-root data/large_model_track_test \
        --out data/cache/large_model_track_test \
        --index data/cache/test_index.csv --frames 16 --workers 4 --modality {MODALITY}

    # (c) 전처리 결과물 → Drive 백업 (다음 세션에서 (a)로 복원됨)
    print('전처리 캐시를 Drive에 백업 중...')
    !tar -cf {CACHE_TAR} data/cache
else:
    print('전처리 캐시 이미 완료 — skip')

# (d) 인덱스 csv는 낱개로도 Drive에 저장 (EDA/팀 공유용)
!cp data/cache/*_index.csv {DRIVE_OUT}/
!ls -lh {DRIVE_OUT}

In [ ]:
# ===== 6. 로컬 검증 (hold-out user 9, 24) =====
# 먼저 --limit 100으로 빠르게 확인하고, 괜찮으면 limit 제거
!python src/run_baseline.py --qa data/training_qa.csv \
    --out {DRIVE_OUT}/val_pred.csv --val-users 9,24 --limit 100 \
    --media-root data/cache,data --model {MODEL} --modality {MODALITY}

!python src/evaluate.py --pred {DRIVE_OUT}/val_pred.csv --gold data/training_qa.csv

In [ ]:
# ===== 7. 테스트 추론 → Drive에 submission 저장 =====
# 세션이 끊겨도 같은 명령을 다시 실행하면 이어서 돌아감 (resume)
!python src/run_baseline.py --qa data/test_qa.csv \
    --out {DRIVE_OUT}/submission.csv \
    --media-root data/cache,data --model {MODEL} --modality {MODALITY}

import pandas as pd
sub = pd.read_csv(f'{DRIVE_OUT}/submission.csv')

# 구버전 헤더(answer)면 Kaggle이 요구하는 'prediction'으로 자동 변환
if 'answer' in sub.columns:
    sub = sub.rename(columns={'answer': 'prediction'})
    sub.to_csv(f'{DRIVE_OUT}/submission.csv', index=False)
    print("헤더를 'prediction'으로 변환했습니다")

# 완성 여부 검사: 전체 문항 수와 일치해야 제출 가능
qa = pd.read_csv('data/test_qa.csv')
if len(sub) == len(qa) and sub['qa_id'].nunique() == len(qa) and 'prediction' in sub.columns:
    print(f'✅ 완료: {len(sub)}/{len(qa)} rows, 컬럼 {list(sub.columns)} — 제출 가능')
else:
    print(f'⚠️ 미완성: {len(sub)}/{len(qa)} rows — 이 셀을 다시 실행해서 이어서 추론하세요 (제출 금지)')
sub.head()

완료 후 Drive의 `cuhk/submission.csv`를 Kaggle에 제출.

**Drive(`cuhk/`)에 저장되는 것들**

| 파일 | 내용 |
|---|---|
| `submission.csv` | 테스트 예측 (Kaggle 제출용) |
| `val_pred.csv` | 로컬 검증 예측 |
| `cache.tar` | 전처리 프레임 캐시 전체 — 다음 세션에서 자동 복원되어 재전처리 생략 |
| `harn_index.csv` / `hau_index.csv` / `test_index.csv` | 클립 인덱스 (EDA/팀 공유용) |

**팁**
- 코드 수정은 로컬 VS Code에서 → push → 여기서는 **셀 2(git pull)부터** 다시 실행
- **`MODALITY`를 바꿔 재전처리하려면** Drive의 `cache.tar`와 인덱스 csv를 삭제(또는 이름 변경) 후 재실행 — 안 지우면 이전 modality 캐시를 그대로 재사용함
- HF 다운로드가 느린 날은: zip들을 Drive에 한 번 복사해 두고, 셀 4를 `Drive → /content 복사 + unzip`으로 교체
- 검증 정확도는 category별로 보고 병목(multi/sequence)부터 개선 (PIPELINE.md 7절 로드맵)